In [2]:
import duckdb
import openml
import pandas as pd
from pathlib import Path

# 1. Setup paths
DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "raw" / "orange_belgium_uplift.parquet"
db_path = DATA_DIR / "igaming_uplift.duckdb"

# Connect to a persistent DuckDB file
con = duckdb.connect(str(db_path))
print("Libraries loaded and DuckDB connected!")

# 2. Ingest Data (Fallback to OpenML if local file is missing)
if RAW_DATA_PATH.exists():
    print(f"Loading local raw data from: {RAW_DATA_PATH}")
    # DuckDB can read Parquet files directly via SQL
    ingest_query = f"SELECT * FROM read_parquet('{RAW_DATA_PATH}')"
else:
    print("Local data not found. Fetching from OpenML...")
    dataset = openml.datasets.get_dataset(45580)
    X, y, _, attribute_names = dataset.get_data(
        dataset_format="dataframe", target=dataset.default_target_attribute
    )
    df_raw = pd.DataFrame(X, columns=attribute_names)
    if y is not None:
        df_raw["churn"] = y
    con.register('df_raw_view', df_raw)
    ingest_query = "SELECT * FROM df_raw_view"

# 3. Perform Type Coercion and Cleaning using SQL
query_clean = f"""
CREATE OR REPLACE TABLE clean_panel AS 
SELECT 
    *,
    TRY_CAST(t AS INTEGER) AS treatment_flag,
    TRY_CAST(churn AS INTEGER) AS target_churn
FROM ({ingest_query});
"""
con.execute(query_clean)

# 4. Verify the transformation
result = con.execute("SELECT treatment_flag, target_churn, COUNT(*) as users FROM clean_panel GROUP BY 1, 2").df()
display(result)

Libraries loaded and DuckDB connected!
Loading local raw data from: ..\data\raw\orange_belgium_uplift.parquet


,treatment_flag,target_churn,users
0,0,0,2781
1,0,1,105
2,1,0,8707
3,1,1,303


In [3]:
from scipy import stats
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Pull the clean data from DuckDB into a Pandas DataFrame
df_clean = con.execute("SELECT * FROM clean_panel").df()

# Define columns
treatment_col = "treatment_flag"
target_col = "target_churn"

# 2. Train / Test Split (Stratified to preserve distributions)
stratify_col = df_clean[treatment_col].astype(str) + "_" + df_clean[target_col].astype(str)
train_df, test_df = train_test_split(
    df_clean, test_size=0.30, random_state=42, stratify=stratify_col
)
print(f"Data Split -> Train: {len(train_df)} users | Test: {len(test_df)} users\n")

# 3. Covariate Balance Audit (Standardized Mean Difference)
# We want to prove the RCT is valid (SMD < 0.10)
print("--- RCT Covariate Balance Audit ---")
treated = train_df[train_df[treatment_col] == 1]
control = train_df[train_df[treatment_col] == 0]

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
covariates = [c for c in numeric_cols if c not in [treatment_col, target_col, "t", "churn"]]

balance_records = []
for col in covariates:
    s_t, s_c = treated[col].dropna(), control[col].dropna()
    m_t, m_c = s_t.mean(), s_c.mean()
    v_t, v_c = s_t.var(), s_c.var()
    
    # Calculate Standardized Mean Difference (SMD)
    pooled_sd = np.sqrt((v_t + v_c) / 2)
    smd = (m_t - m_c) / pooled_sd if pooled_sd > 0 else 0.0
    
    balance_records.append({
        "Covariate": col,
        "Abs_SMD": round(abs(smd), 4)
    })

balance_df = pd.DataFrame(balance_records).sort_values("Abs_SMD", ascending=False)
max_smd = balance_df["Abs_SMD"].max()

print(f"Total Features Checked: {len(covariates)}")
print(f"Maximum Absolute SMD: {max_smd:.4f}")
if max_smd < 0.10:
    print("✅ SUCCESS: Data is well-balanced (Max SMD < 0.10). RCT assumptions hold.")
else:
    print("⚠️ WARNING: Potential imbalance detected (SMD > 0.10).")

# Display the top 5 most imbalanced features
display(balance_df.head(5))

Data Split -> Train: 8327 users | Test: 3569 users

--- RCT Covariate Balance Audit ---
Total Features Checked: 160
Maximum Absolute SMD: 0.2361
⚠️ WARNING: Potential imbalance detected (SMD > 0.10).


,Covariate,Abs_SMD
7,PC8,0.2361
27,PC28,0.1775
0,PC1,0.1228
3,PC4,0.1219
85,PC86,0.1132


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# 1. Define Features and Target Arrays
feature_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c not in [treatment_col, target_col, "t", "churn"]]

X_tr, W_tr, y_tr = train_df[feature_cols].values, train_df[treatment_col].values, train_df[target_col].values
X_te, W_te, y_te = test_df[feature_cols].values, test_df[treatment_col].values, test_df[target_col].values

print("Training Causal Meta-Learners (This might take ~15-30 seconds)...")

# 2. Train T-Learner (Fits separate models for Treatment and Control)
t_model_0 = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1)
t_model_1 = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1)

t_model_0.fit(X_tr[W_tr == 0], y_tr[W_tr == 0])
t_model_1.fit(X_tr[W_tr == 1], y_tr[W_tr == 1])

# T-Learner Conditional Average Treatment Effect (CATE)
tau_t = t_model_1.predict(X_te) - t_model_0.predict(X_te)

# 3. Train X-Learner (Cross-Imputes effects and corrects for our data imbalance)
# Stage 2: Calculate imputed counterfactual effects using T-Learner predictions
d_1 = y_tr[W_tr == 1] - t_model_0.predict(X_tr[W_tr == 1])
d_0 = t_model_1.predict(X_tr[W_tr == 0]) - y_tr[W_tr == 0]

tau_1_model = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1)
tau_0_model = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1)
tau_1_model.fit(X_tr[W_tr == 1], d_1)
tau_0_model.fit(X_tr[W_tr == 0], d_0)

# Stage 3: Propensity model to weight and correct the RCT imbalance we found!
propensity_model = LogisticRegression(max_iter=1000, random_state=42)
propensity_model.fit(X_tr, W_tr)
e_hat = propensity_model.predict_proba(X_te)[:, 1]

# X-Learner CATE
tau_x = e_hat * tau_0_model.predict(X_te) + (1.0 - e_hat) * tau_1_model.predict(X_te)
print("Training Complete!\n")

# 4. Compare Meta-Learner Predictions
# Note: A negative tau means the treatment REDUCED churn (which is good!)
results_df = pd.DataFrame({
    "Estimator": ["T-Learner", "X-Learner"],
    "Mean CATE (Overall Churn Impact)": [tau_t.mean(), tau_x.mean()],
    "% Persuadable (Treatment Decreased Churn)": [(tau_t < -0.001).mean() * 100, (tau_x < -0.001).mean() * 100],
    "% Sleeping Dog (Treatment Increased Churn)": [(tau_t > 0.001).mean() * 100, (tau_x > 0.001).mean() * 100]
}).round(2)

display(results_df)

Training Causal Meta-Learners (This might take ~15-30 seconds)...
Training Complete!



,Estimator,Mean CATE (Overall Churn Impact),% Persuadable (Treatment Decreased Churn),% Sleeping Dog (Treatment Increased Churn)
0,T-Learner,-0.01,45.95,49.73
1,X-Learner,-0.01,43.12,52.62


In [6]:
# 1. Economic Constants
LTV = 250.0       # Value of a retained player (€)
BONUS_COST = 5.0  # Cost of retention offer/bonus (€)

# We only target users if their expected churn reduction (-CATE) is greater than the cost ratio
ROI_THRESHOLD = BONUS_COST / LTV 

# 2. Define Business Policies
# Policy A: Mass Outreach (Give the bonus to everyone)
flag_mass = np.ones(len(y_te), dtype=int)

# Policy B: Causal Targeting (Give bonus ONLY if X-Learner predicts profitable uplift)
flag_causal = (-tau_x > ROI_THRESHOLD).astype(int)

def evaluate_policy_pnl(treatment_flags, y_true, w_true, ltv, unit_cost):
    """Computes expected net incremental profit via Horvitz-Thompson IPW."""
    n_targeted = int(np.sum(treatment_flags))
    total_cost = n_targeted * unit_cost
    if n_targeted == 0: return n_targeted, 0.0, 0.0, 0.0, 0.0
    
    mask = (treatment_flags == 1)
    p_w = float(np.mean(w_true))
    y_retained = 1.0 - y_true
    
    # Inverse Probability Weighting (IPW) to estimate true incremental retention
    ret_t = np.sum(y_retained[mask & (w_true == 1)]) / p_w
    ret_c = np.sum(y_retained[mask & (w_true == 0)]) / (1.0 - p_w)
    net_retained = float(ret_t - ret_c)
    
    gross_rev = net_retained * ltv
    net_profit = gross_rev - total_cost
    roi_pct = (net_profit / total_cost) * 100.0 if total_cost > 0 else 0.0
    return n_targeted, net_retained, total_cost, net_profit, roi_pct

# 3. Simulate P&L Outcomes
pnl_mass = evaluate_policy_pnl(flag_mass, y_te, W_te, LTV, BONUS_COST)
pnl_causal = evaluate_policy_pnl(flag_causal, y_te, W_te, LTV, BONUS_COST)

# 4. Generate Executive Summary Table
pnl_df = pd.DataFrame({
    "Targeting Policy": ["1. Mass Outreach (100%)", "2. Causal Uplift (X-Learner Optimal)"],
    "Users Contacted": [f"{pnl_mass[0]} (100.0%)", f"{pnl_causal[0]} ({(pnl_causal[0]/len(y_te)*100):.1f}%)"],
    "Incremental Players Retained": [f"{pnl_mass[1]:.1f}", f"{pnl_causal[1]:.1f}"],
    "Total Campaign Cost": [f"€{pnl_mass[2]:,.2f}", f"€{pnl_causal[2]:,.2f}"],
    "Net Profit": [f"€{pnl_mass[3]:,.2f}", f"€{pnl_causal[3]:,.2f}"],
    "Campaign ROI": [f"{pnl_mass[4]:.1f}%", f"{pnl_causal[4]:.1f}%"]
})

display(pnl_df)

,Targeting Policy,Users Contacted,Incremental Players Retained,Total Campaign Cost,Net Profit,Campaign ROI
0,1. Mass Outreach (100%),3569 (100.0%),11.7,"€17,845.00","€-14,913.78",-83.6%
1,2. Causal Uplift (X-Learner Optimal),688 (19.3%),108.3,"€3,440.00","€23,636.27",687.1%


In [7]:
import numpy as np

# 1. Compile the dashboard dataset
powerbi_df = pd.DataFrame({
    "Treatment_Flag": W_te,
    "Actual_Churn": y_te,
    "X_Learner_CATE": tau_x.round(4)
})

# 2. Map the CATE scores to their Business Quadrant
# Note: Negative CATE means treatment reduced churn
conditions = [
    (powerbi_df["X_Learner_CATE"] < -ROI_THRESHOLD),  # Profitable to treat
    (powerbi_df["X_Learner_CATE"] > 0.001)            # Backfires if treated
]
choices = ["Persuadable", "Sleeping Dog"]
powerbi_df["Predicted_Quadrant"] = np.select(conditions, choices, default="Sure Thing / Lost Cause")

# 3. Export to CSV for Power BI
OUTPUTS_DIR = Path("../outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

export_path = OUTPUTS_DIR / "powerbi_causal_export.csv"
powerbi_df.to_csv(export_path, index=False)

print(f"✅ Dashboard dataset exported successfully: {export_path}")
display(powerbi_df.head())

✅ Dashboard dataset exported successfully: ..\outputs\powerbi_causal_export.csv


,Treatment_Flag,Actual_Churn,X_Learner_CATE,Predicted_Quadrant
0,1,0,-0.0274,Persuadable
1,1,0,0.0059,Sleeping Dog
2,0,0,0.0071,Sleeping Dog
3,0,0,0.0005,Sure Thing / Lost Cause
4,1,0,0.0297,Sleeping Dog
